# yolo26-analytics Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohibShaikh/yolo26-analytics/blob/main/notebooks/demo.ipynb)

This notebook demonstrates the core features of **yolo26-analytics**:
1. Object detection with YOLO26
2. Multi-object tracking with ByteTrack
3. Zone analytics (counting, entry/exit, dwell time)
4. Heatmap generation
5. SAHI for small object detection

## 1. Install

In [ ]:
!pip install -q yolo26-analytics opencv-python-headless matplotlib

## 2. Download a sample video

In [ ]:
import urllib.request
from pathlib import Path

VIDEO_URL = "https://ultralytics.com/assets/people-walking.mp4"
VIDEO_PATH = "people-walking.mp4"

if not Path(VIDEO_PATH).exists():
    print("Downloading sample video...")
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
    print("Done.")
else:
    print("Video already exists.")

## 3. Basic Detection + Tracking

Run YOLO26 detection and ByteTrack tracking on a few frames, then visualize.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from yolo26_analytics.detection.yolo26 import YOLO26Detector
from yolo26_analytics.tracking.bytetrack import ByteTrackAdapter

detector = YOLO26Detector(weights="yolo26n.pt", confidence=0.4)
tracker = ByteTrackAdapter(max_age=30, min_hits=3)

cap = cv2.VideoCapture(VIDEO_PATH)
frames_to_show = []

for i in range(60):
    ret, frame = cap.read()
    if not ret:
        break

    detections = detector.predict(frame)
    tracks = tracker.update(detections)

    # Draw bboxes and track IDs
    for track in tracks:
        x1, y1, x2, y2 = track.bbox
        color = (
            (track.track_id * 50) % 255,
            (track.track_id * 80) % 255,
            (track.track_id * 120) % 255,
        )
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        label = f"ID:{track.track_id} {track.class_name} {track.confidence:.2f}"
        cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    if i % 20 == 0:
        frames_to_show.append(frame.copy())

cap.release()

fig, axes = plt.subplots(1, len(frames_to_show), figsize=(6 * len(frames_to_show), 6))
if len(frames_to_show) == 1:
    axes = [axes]
for ax, f in zip(axes, frames_to_show, strict=False):
    ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    ax.axis("off")
plt.suptitle("Detection + Tracking (frames 0, 20, 40)")
plt.tight_layout()
plt.show()

print(f"Detected {len(tracks)} tracks in last frame")

## 4. Zone Analytics

Define a zone polygon and run counting + entry/exit detection.

In [ ]:
from yolo26_analytics.config.schema import ZoneAnalyticsRule, ZoneConfig
from yolo26_analytics.zones.analyzer import ZoneAnalyzer

# Get frame dimensions
cap = cv2.VideoCapture(VIDEO_PATH)
ret, sample_frame = cap.read()
h, w = sample_frame.shape[:2]
cap.release()

# Define a zone covering the right half of the frame
zone_cfg = ZoneConfig(
    name="Right Zone",
    polygon=[[w // 2, 0], [w, 0], [w, h], [w // 2, h]],
    track_classes=["person"],
    analytics=[
        ZoneAnalyticsRule(type="count"),
        ZoneAnalyticsRule(type="entry_exit"),
        ZoneAnalyticsRule(type="dwell", alert_threshold=5),
    ],
    cooldown=2,
)

zone_analyzer = ZoneAnalyzer([zone_cfg])

# Reset tracker and run through video
tracker.reset()
cap = cv2.VideoCapture(VIDEO_PATH)
all_events = []

for _i in range(120):
    ret, frame = cap.read()
    if not ret:
        break

    detections = detector.predict(frame)
    tracks = tracker.update(detections)
    events = zone_analyzer.check(tracks)
    all_events.extend(events)

cap.release()

print(f"Total events detected: {len(all_events)}")
for event in all_events[:10]:
    print(
        f"  [{event.event_type}] zone={event.zone_name}"
        f" track_id={event.track_id} class={event.object_class}"
    )

In [ ]:
# Visualize the zone on a frame
cap = cv2.VideoCapture(VIDEO_PATH)
ret, vis_frame = cap.read()
cap.release()

zone = zone_analyzer.zones[0]
pts = np.array(zone_cfg.polygon, dtype=np.int32)
cv2.polylines(vis_frame, [pts], True, (0, 255, 0), 3)
cv2.putText(
    vis_frame, zone_cfg.name, (pts[0][0] + 10, pts[0][1] + 30),
    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2,
)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
plt.title(f"Zone: {zone_cfg.name} | Events: {len(all_events)}")
plt.axis("off")
plt.show()

## 5. Heatmap Generation

Accumulate track centroids and generate a heatmap overlay.

In [ ]:
from yolo26_analytics.analytics.heatmap import HeatmapAccumulator, generate_heatmap_image

cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
h, w = first_frame.shape[:2]

heatmap_acc = HeatmapAccumulator(width=w, height=h)
tracker.reset()

frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    detections = detector.predict(frame)
    tracks = tracker.update(detections)

    for track in tracks:
        cx, cy = track.centroid
        heatmap_acc.add_point(cx, cy)

    frame_count += 1

cap.release()

# Generate overlay
overlay = generate_heatmap_image(heatmap_acc, first_frame, alpha=0.6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
ax1.set_title("Original Frame")
ax1.axis("off")
ax2.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
ax2.set_title(f"Heatmap ({frame_count} frames)")
ax2.axis("off")
plt.tight_layout()
plt.show()

# Also show raw heatmap
raw = heatmap_acc.get_heatmap()
plt.figure(figsize=(10, 6))
plt.imshow(raw, cmap="hot", interpolation="gaussian")
plt.colorbar(label="Track density")
plt.title("Raw Heatmap (centroid accumulation)")
plt.axis("off")
plt.show()

## 6. SAHI — Small Object Detection

Compare standard detection vs SAHI (sliced inference) on the same frame.

In [ ]:
from yolo26_analytics.detection.sahi import SAHIDetector

cap = cv2.VideoCapture(VIDEO_PATH)
ret, test_frame = cap.read()
cap.release()

# Standard detection
standard_dets = detector.predict(test_frame)

# SAHI detection (smaller slices = better for small objects)
sahi_detector = SAHIDetector(detector, slice_size=320, overlap=0.25)
sahi_dets = sahi_detector.predict(test_frame)

def draw_dets(frame, dets, color, label_prefix):
    vis = frame.copy()
    for d in dets:
        x1, y1, x2, y2 = d.bbox
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            vis, f"{label_prefix} {d.class_name} {d.confidence:.2f}",
            (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1,
        )
    return vis

vis_standard = draw_dets(test_frame, standard_dets, (0, 255, 0), "STD")
vis_sahi = draw_dets(test_frame, sahi_dets, (255, 0, 0), "SAHI")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(cv2.cvtColor(vis_standard, cv2.COLOR_BGR2RGB))
ax1.set_title(f"Standard Detection ({len(standard_dets)} objects)")
ax1.axis("off")
ax2.imshow(cv2.cvtColor(vis_sahi, cv2.COLOR_BGR2RGB))
ax2.set_title(f"SAHI Detection ({len(sahi_dets)} objects)")
ax2.axis("off")
plt.tight_layout()
plt.show()

## 7. Full Pipeline (YAML Config)

Build and run the full async pipeline from a YAML config string.

In [ ]:
from pathlib import Path

config_yaml = f"""
source:
  type: video_file
  path: {VIDEO_PATH}

model:
  weights: yolo26n.pt
  confidence: 0.4

tracking:
  engine: bytetrack
  max_age: 30
  min_hits: 3

store:
  type: sqlite
  path: ./demo_data/analytics.db

zones:
  - name: "Monitor Zone"
    polygon: [[0, 0], [{w}, 0], [{w}, {h}], [0, {h}]]
    track_classes: [person]
    analytics:
      - type: count
      - type: entry_exit
    cooldown: 5

alerts:
  - type: console
"""

Path("demo_config.yaml").write_text(config_yaml)
print("Config written to demo_config.yaml")
print()
print(config_yaml)

In [ ]:
from yolo26_analytics.core.pipeline import Pipeline

pipeline = Pipeline.from_yaml("demo_config.yaml")

# Run the pipeline (processes the full video)

await pipeline.run_async()

print("\nPipeline complete!")
print(f"  Frames processed: {pipeline.frame_count}")
print(f"  Final FPS: {pipeline.fps:.1f}")
print(f"  Last latency: {pipeline.last_latency_ms:.1f}ms")

## 8. Custom Detector (Protocol Pattern)

Any class with `predict(frame) -> list[Detection]` works — no inheritance needed.

In [ ]:
from yolo26_analytics.models import Detection
from yolo26_analytics.protocols import Detector


class CenterDetector:
    """Dummy detector that always detects an object in the center."""

    def predict(self, frame):
        h, w = frame.shape[:2]
        return [
            Detection(
                bbox=(w // 4, h // 4, 3 * w // 4, 3 * h // 4),
                confidence=0.99,
                class_name="custom_object",
            )
        ]


# Verify it satisfies the Detector protocol
assert isinstance(CenterDetector(), Detector)
print("CenterDetector satisfies the Detector protocol!")

# Use it
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()

custom_det = CenterDetector()
results = custom_det.predict(frame)
print(f"Custom detector found: {results}")

## Summary

**yolo26-analytics** provides:

| Feature | What it does |
|---|---|
| `YOLO26Detector` | Pluggable YOLO26 inference adapter |
| `ByteTrackAdapter` | Multi-object tracking with persistent IDs |
| `ZoneAnalyzer` | Counting, entry/exit, dwell time, throughput |
| `HeatmapAccumulator` | Centroid-based heatmap generation |
| `SAHIDetector` | Slice-based inference for small objects |
| `Pipeline` | Full async orchestrator with YAML config |
| `AlertManager` | Console, webhook, MQTT, Telegram alerts |
| Dashboard | FastAPI + HTMX live view |
| CLI (`y26a`) | Run, export, heatmap commands |

```bash
pip install yolo26-analytics
```

GitHub: [MohibShaikh/yolo26-analytics](https://github.com/MohibShaikh/yolo26-analytics)